In [27]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge

from skrub import TableReport

In [28]:
file_path = Path("/home/joffreyma/TS/Code/bike_counters_skore/submissions/external_data/external_data.csv")
df_ext = pd.read_csv(file_path, parse_dates=["date"])

In [29]:
TableReport(df_ext)

Processing column  59 / 59


<TableReport: use .open() to display>

In [30]:
df_ext[["date", "t"]].head(5)

,date,t
0,2021-01-01 00:00:00,272.75
1,2021-01-01 03:00:00,271.25
2,2021-01-01 06:00:00,271.95
3,2021-01-01 09:00:00,272.45
4,2021-01-01 12:00:00,276.95


In [31]:
data = pd.read_parquet(Path("data") / "train.parquet").sort_values(by="date")

In [32]:
data.head()

,counter_id,counter_name,site_id,site_name,bike_count,date,counter_installation_date,counter_technical_id,latitude,longitude,log_bike_count
705677,100056332-104056332,Pont de Bercy SO-NE,100056332,Pont de Bercy,0.0,2020-09-01 01:00:00,2019-12-11,Y2H19070378,48.83848,2.37587,0.000000
333389,100047547-104047547,6 rue Julia Bartet NE-SO,100047547,6 rue Julia Bartet,4.0,2020-09-01 01:00:00,2018-11-28,Y2H18086323,48.82636,2.30303,1.609438
343292,100047547-103047547,6 rue Julia Bartet SO-NE,100047547,6 rue Julia Bartet,2.0,2020-09-01 01:00:00,2018-11-28,Y2H18086323,48.82636,2.30303,1.098612
805911,100057380-103057380,Totem Cours la Reine O-E,100057380,Totem Cours la Reine,0.0,2020-09-01 01:00:00,2020-02-11,YTH19111509,48.86462,2.31444,0.000000
353162,100047548-103047548,Face au 25 quai de l'Oise NE-SO,100047548,Face au 25 quai de l'Oise,2.0,2020-09-01 01:00:00,2018-11-28,Y2H18086324,48.89141,2.38482,1.098612


In [33]:
data = data.copy()
# When using merge_asof left frame need to be sorted
data["orig_index"] = np.arange(data.shape[0])

In [34]:
df_ext["date"] = df_ext["date"].astype('datetime64[ms]')

In [35]:
data["date"] = data["date"].astype('datetime64[ms]')

In [36]:
merged_data = pd.merge_asof(
        data.sort_values(by="date"), df_ext[["date", "t"]].sort_values("date"), on="date"
    )